# Lesson 12: k-means clustering of 37 economies, k = 2 with the highest mean silhouette

An economy is described by its mean policy rate of 2025, its mean CPI inflation of 2025 and the change of its policy rate from December 2021 to December 2025, all in per cent. Türkiye, at a policy rate of 42.83 per cent and an inflation of 35.18 per cent, is far from all other economies and is set apart as a group of its own. The 37 other economies are standardised to mean 0 and sd 1 on each feature. In k-means, an economy is in the group of its nearest centroid, and a centroid is the mean of the economies in its group. Among the numbers of groups from 2 to 6, the mean silhouette is highest at k = 2, and k = 2 is also the elbow of the curve of the within-cluster sum of squares (WCSS).

Policy rates and consumer prices: central banks and national statistical offices via the BIS Data Portal, Bank for International Settlements. Snapshot 30 September 2026.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

economies = pd.read_csv("https://digital-ai-finance.github.io/teaching-msc-data-science/data/2026-09-30-bis-economies.csv")
rest = economies[economies.code != "TR"]  # Türkiye, at a policy rate of 42.83 per cent, is a group of its own
features = ["policy_rate_2025", "cpi_inflation_2025", "policy_change_2021_2025"]
scaler = StandardScaler().fit(rest[features])
z = scaler.transform(rest[features])
print(f"{len(rest)} economies, policy rate mean {scaler.mean_[0]:.2f}, sd {scaler.scale_[0]:.2f} per cent")

two = KMeans(n_clusters=2, n_init=10, random_state=42).fit(z)  # k = 2, the decision of the next cell
seven = two.labels_ == two.cluster_centers_[:, 0].argmax()  # the group with the higher policy rate
centres = scaler.inverse_transform(two.cluster_centers_)
plt.scatter(rest.policy_rate_2025[seven], rest.cpi_inflation_2025[seven], color="tab:red", label=f"group of {seven.sum()}")
plt.scatter(rest.policy_rate_2025[~seven], rest.cpi_inflation_2025[~seven], color="tab:blue", label=f"group of {(~seven).sum()}")
plt.scatter(centres[:, 0], centres[:, 1], marker="x", s=120, color="black", label="centroid of a group")
plt.xlabel("mean policy rate 2025 (per cent)")
plt.ylabel("mean CPI inflation 2025 (per cent)")
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.show()

In [ ]:
k = 2  # the elbow of the WCSS curve, the highest mean silhouette of k from 2 to 6
model = KMeans(n_clusters=k, n_init=10, random_state=42).fit(z)
print(f"mean silhouette at k = {k}: {silhouette_score(z, model.labels_):.2f}")
print("economies per group:", *sorted(np.bincount(model.labels_)))
print(f"highest policy rate of a centroid: {scaler.inverse_transform(model.cluster_centers_)[:, 0].max():.2f} per cent")

Change `k = 2` to `k = 3` and run the cell again. At k = 3 the mean silhouette is 0.36, down from 0.52 at k = 2.